In [1]:
from pathlib import Path

print("Local notebook mode")
print("cwd:", Path.cwd().resolve())


Local notebook mode
cwd: C:\Users\USER\Desktop\chess_engine\train


In [2]:
import numpy as np
import os
import sys
import random
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model").exists() and (candidate / "train").exists() and (candidate / "runs").exists():
            return candidate
    raise RuntimeError(f"Cannot find repo root from: {start}")


REPO_ROOT = find_repo_root(Path.cwd())
MODEL_ROOT = REPO_ROOT / "model"
RUNS_ROOT = REPO_ROOT / "runs"
IS_WINDOWS = (os.name == "nt")

if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))
    print(f"Added to sys.path: {MODEL_ROOT}")
else:
    print(f"Already in sys.path: {MODEL_ROOT}")

architecture_folder_path = MODEL_ROOT / "architecture_v2"
required_files = ["model.py", "blocks.py", "head.py"]
for file_name in required_files:
    file_path = architecture_folder_path / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required architecture file: {file_path}")

from architecture_v2.model import DGRNChessNetV2 as DGRNChessNet


SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"RUNS_ROOT = {RUNS_ROOT}")
print(f"Global seed = {SEED}")
print(f"Imported model class = {getattr(DGRNChessNet, '__name__', DGRNChessNet)}")


Added to sys.path: C:\Users\USER\Desktop\chess_engine\model
REPO_ROOT = C:\Users\USER\Desktop\chess_engine
RUNS_ROOT = C:\Users\USER\Desktop\chess_engine\runs
Global seed = 123
Imported model class = DGRNChessNetV2


In [3]:
from pathlib import Path
import numpy as np

DATA_ROOT = REPO_ROOT / "data" / "process"
DATA_ROOT_ACTIVE = DATA_ROOT
USE_LOCAL_CACHE = False
SPLITS = ("train", "val", "test")

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Missing local data root: {DATA_ROOT}")


def _sorted_npy_files(split_dir: Path, pattern: str):
    return sorted(split_dir.glob(pattern), key=lambda p: p.name)


def scan_split(split: str):
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Missing split dir: {split_dir}")

    x_files = _sorted_npy_files(split_dir, "X_*.npy")
    y_files = _sorted_npy_files(split_dir, "y_*.npy")
    if len(x_files) == 0 or len(y_files) == 0:
        raise FileNotFoundError(f"No shards found in {split_dir}. Expect X_*.npy and y_*.npy")

    if len(x_files) != len(y_files):
        raise ValueError(f"Shard count mismatch in {split}: X={len(x_files)} vs y={len(y_files)}")

    shard_sizes = []
    for xf, yf in zip(x_files, y_files):
        X = np.load(xf, mmap_mode="r")
        y = np.load(yf, mmap_mode="r")

        if X.ndim != 4 or X.shape[1:] != (18, 8, 8):
            raise ValueError(f"Bad X shape at {xf.name}: {X.shape} (expect (N,18,8,8))")
        if y.ndim != 1:
            raise ValueError(f"Bad y shape at {yf.name}: {y.shape} (expect (N,))")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"N mismatch at {xf.name} vs {yf.name}: {X.shape[0]} vs {y.shape[0]}")

        if X.dtype != np.uint8:
            print(f"[warn] {split}/{xf.name}: X dtype is {X.dtype}, expect uint8")
        if y.dtype not in (np.float16, np.float32):
            print(f"[warn] {split}/{yf.name}: y dtype is {y.dtype}, expect float16/float32")

        shard_sizes.append(int(X.shape[0]))

    shard_sizes = np.array(shard_sizes, dtype=np.int64)
    offsets = np.zeros(len(shard_sizes) + 1, dtype=np.int64)
    offsets[1:] = np.cumsum(shard_sizes)

    return {
        "split": split,
        "x_files": [str(p) for p in x_files],
        "y_files": [str(p) for p in y_files],
        "shard_sizes": shard_sizes,
        "offsets": offsets,
        "num_shards": len(shard_sizes),
        "num_samples": int(offsets[-1]),
    }


SHARDS = {split: scan_split(split) for split in SPLITS}

print(f"DATA_ROOT = {DATA_ROOT}")
for split in SPLITS:
    meta = SHARDS[split]
    print(
        f"{split}: shards={meta['num_shards']} "
        f"samples={meta['num_samples']} "
        f"first={Path(meta['x_files'][0]).name} "
        f"last={Path(meta['x_files'][-1]).name}"
    )


DATA_ROOT = C:\Users\USER\Desktop\chess_engine\data\process
train: shards=80 samples=4000000 first=X_00000.npy last=X_00079.npy
val: shards=10 samples=500000 first=X_00000.npy last=X_00009.npy
test: shards=10 samples=500000 first=X_00000.npy last=X_00009.npy


In [4]:
# ===== Cell: Dataset + DataLoader (local profile) =====
import numpy as np
import torch
import random
from torch.utils.data import Dataset, DataLoader


class ShardedNpyDataset(Dataset):
    """
    Map-style dataset for X_*.npy (uint8, N,18,8,8) and y_*.npy (float16/32, N).
    """

    def __init__(self, meta: dict, dtype_y=torch.float32, use_mmap=True):
        self.meta = meta
        self.x_files = meta["x_files"]
        self.y_files = meta["y_files"]
        self.offsets = np.asarray(meta["offsets"], dtype=np.int64)
        self.num_samples = int(meta["num_samples"])
        self.num_shards = int(meta["num_shards"])
        self.dtype_y = dtype_y
        self.use_mmap = bool(use_mmap)
        self._X = [None] * self.num_shards
        self._y = [None] * self.num_shards

    def __len__(self):
        return self.num_samples

    def _open_shard_if_needed(self, shard_id: int):
        if self._X[shard_id] is None:
            self._X[shard_id] = np.load(self.x_files[shard_id], mmap_mode="r" if self.use_mmap else None)
        if self._y[shard_id] is None:
            self._y[shard_id] = np.load(self.y_files[shard_id], mmap_mode="r" if self.use_mmap else None)

    def __getitem__(self, idx):
        g = int(idx)
        shard_id = int(np.searchsorted(self.offsets, g, side="right") - 1)
        local_i = int(g - self.offsets[shard_id])
        self._open_shard_if_needed(shard_id)
        X = self._X[shard_id][local_i]
        y = self._y[shard_id][local_i]
        X = torch.from_numpy(X)
        y = torch.as_tensor(y, dtype=self.dtype_y)
        return X, y


class ShardLocalBatchSampler:
    """
    Batch sampler that keeps shard locality while preserving enough randomness.
    """

    def __init__(
        self,
        meta: dict,
        batch_size: int,
        drop_last: bool = False,
        seed: int = 123,
        shuffle_shards: bool = True,
        local_shuffle_block: int = 16384,
        shuffle_within_block: bool = True,
        shuffle_block_order: bool = True,
    ):
        self.offsets = np.asarray(meta["offsets"], dtype=np.int64)
        self.shard_sizes = np.asarray(meta["shard_sizes"], dtype=np.int64)
        self.num_samples = int(meta["num_samples"])
        self.num_shards = int(meta["num_shards"])
        self.batch_size = int(batch_size)
        self.drop_last = bool(drop_last)
        self.seed = int(seed)
        self.shuffle_shards = bool(shuffle_shards)
        self.local_shuffle_block = int(local_shuffle_block)
        self.shuffle_within_block = bool(shuffle_within_block)
        self.shuffle_block_order = bool(shuffle_block_order)
        self.epoch = 0
        self.start_batch = 0

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def set_start_batch(self, start_batch: int):
        self.start_batch = max(0, int(start_batch))

    def __len__(self):
        if self.drop_last:
            return self.num_samples // self.batch_size
        return (self.num_samples + self.batch_size - 1) // self.batch_size

    def _local_order(self, n: int, rng: np.random.Generator):
        idx = np.arange(n, dtype=np.int64)
        block_size = max(1, self.local_shuffle_block)
        if block_size <= 1:
            rng.shuffle(idx)
            return idx
        n_blocks = (n + block_size - 1) // block_size
        blocks = np.arange(n_blocks, dtype=np.int64)
        rng.shuffle(blocks)
        out = np.empty(n, dtype=np.int64)
        pos = 0
        for block_id in blocks:
            start = int(block_id * block_size)
            end = min(n, start + block_size)
            block = idx[start:end].copy()
            if self.shuffle_within_block:
                rng.shuffle(block)
            span = end - start
            out[pos:pos + span] = block
            pos += span
        return out

    def _build_global_blocks(self, rng: np.random.Generator):
        shard_order = np.arange(self.num_shards, dtype=np.int64)
        if self.shuffle_shards:
            rng.shuffle(shard_order)
        block_len = max(self.batch_size, self.local_shuffle_block)
        blocks = []
        for shard_id in shard_order:
            shard_id = int(shard_id)
            n = int(self.shard_sizes[shard_id])
            if n <= 0:
                continue
            start = int(self.offsets[shard_id])
            local_idx = self._local_order(n, rng)
            for local_start in range(0, n, block_len):
                local_end = min(n, local_start + block_len)
                block = local_idx[local_start:local_end]
                if block.size > 0:
                    blocks.append(start + block)
        if self.shuffle_block_order:
            rng.shuffle(blocks)
        return blocks

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        blocks = self._build_global_blocks(rng)
        carry = []
        batch_idx = 0
        for block in blocks:
            block_list = block.tolist()
            i = 0
            n = len(block_list)
            while i < n:
                need = self.batch_size - len(carry)
                j = min(i + need, n)
                carry.extend(block_list[i:j])
                i = j
                if len(carry) == self.batch_size:
                    if batch_idx >= self.start_batch:
                        yield carry
                    batch_idx += 1
                    carry = []
        if (not self.drop_last) and len(carry) > 0:
            if batch_idx >= self.start_batch:
                yield carry


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(
    ds: Dataset,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    device: torch.device,
    seed: int,
    persistent_workers: bool = True,
    batch_sampler=None,
    prefetch_factor: int = 2,
):
    pin = (device.type == "cuda")
    generator = torch.Generator()
    generator.manual_seed(seed)
    common = dict(
        dataset=ds,
        num_workers=num_workers,
        pin_memory=pin,
        persistent_workers=(persistent_workers and num_workers > 0),
        worker_init_fn=seed_worker if num_workers > 0 else None,
        generator=generator,
    )
    if num_workers > 0:
        common["prefetch_factor"] = int(prefetch_factor)
    if batch_sampler is not None:
        return DataLoader(batch_sampler=batch_sampler, **common)
    return DataLoader(batch_size=batch_size, shuffle=shuffle, drop_last=False, **common)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

if device.type == "cuda":
    gpu_props = torch.cuda.get_device_properties(0)
    GPU_NAME = gpu_props.name
    GPU_TOTAL_MEM_GB = gpu_props.total_memory / (1024 ** 3)
else:
    GPU_NAME = None
    GPU_TOTAL_MEM_GB = 0.0

print("GPU_NAME:", GPU_NAME)
print("GPU_TOTAL_MEM_GB:", round(GPU_TOTAL_MEM_GB, 2))

train_ds = ShardedNpyDataset(SHARDS["train"], dtype_y=torch.float32, use_mmap=True)
val_ds = ShardedNpyDataset(SHARDS["val"], dtype_y=torch.float32, use_mmap=True)
test_ds = ShardedNpyDataset(SHARDS["test"], dtype_y=torch.float32, use_mmap=True)

if device.type == "cuda":
    if GPU_TOTAL_MEM_GB <= 4.5:
        BATCH_SIZE = 128
        EVAL_BATCH_SIZE = 256
        GRAD_ACCUM_STEPS_DEFAULT = 16
    elif GPU_TOTAL_MEM_GB <= 8.5:
        BATCH_SIZE = 256
        EVAL_BATCH_SIZE = 512
        GRAD_ACCUM_STEPS_DEFAULT = 8
    else:
        BATCH_SIZE = 512
        EVAL_BATCH_SIZE = 1024
        GRAD_ACCUM_STEPS_DEFAULT = 4
else:
    BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    GRAD_ACCUM_STEPS_DEFAULT = 1

NUM_WORKERS = 0 if IS_WINDOWS else (2 if device.type == "cuda" else 1)
LOCAL_SHUFFLE_BLOCK = 32768
TRAIN_DROP_LAST = True

print("TRAIN_MICRO_BATCH_SIZE:", BATCH_SIZE)
print("EVAL_BATCH_SIZE:", EVAL_BATCH_SIZE)
print("GRAD_ACCUM_STEPS_DEFAULT:", GRAD_ACCUM_STEPS_DEFAULT)
print("NUM_WORKERS:", NUM_WORKERS)
if IS_WINDOWS:
    print("[info] Windows + mmap dataset: default NUM_WORKERS=0 because local benchmark showed lower worker count is faster.")

train_batch_sampler = ShardLocalBatchSampler(
    SHARDS["train"],
    batch_size=BATCH_SIZE,
    drop_last=TRAIN_DROP_LAST,
    seed=SEED + 100,
    shuffle_shards=True,
    local_shuffle_block=LOCAL_SHUFFLE_BLOCK,
    shuffle_within_block=True,
    shuffle_block_order=True,
)

train_loader = make_loader(
    train_ds,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    device=device,
    seed=SEED + 1,
    batch_sampler=train_batch_sampler,
)
val_loader = make_loader(val_ds, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device, seed=SEED + 2)
test_loader = make_loader(test_ds, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device, seed=SEED + 3)

xb, yb = next(iter(train_loader))
print("batch X:", xb.shape, xb.dtype, "batch y:", yb.shape, yb.dtype)
print("X min/max:", float(xb.min()), float(xb.max()), "y min/max:", float(yb.min()), float(yb.max()))
print("train micro-batches/epoch:", len(train_loader), "LOCAL_SHUFFLE_BLOCK:", LOCAL_SHUFFLE_BLOCK, "TRAIN_DROP_LAST:", TRAIN_DROP_LAST)


device: cuda
GPU_NAME: NVIDIA GeForce RTX 2050
GPU_TOTAL_MEM_GB: 4.0
TRAIN_MICRO_BATCH_SIZE: 128
EVAL_BATCH_SIZE: 256
GRAD_ACCUM_STEPS_DEFAULT: 16
NUM_WORKERS: 0
[info] Windows + mmap dataset: default NUM_WORKERS=0 because local benchmark showed lower worker count is faster.


C:\Users\USER\AppData\Local\Temp\ipykernel_9860\1747713831.py:41: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  X = torch.from_numpy(X)


batch X: torch.Size([128, 18, 8, 8]) torch.uint8 batch y: torch.Size([128]) torch.float32
X min/max: 0.0 1.0 y min/max: -1.0 0.9286285638809204
train micro-batches/epoch: 31250 LOCAL_SHUFFLE_BLOCK: 32768 TRAIN_DROP_LAST: True


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

Training on device: cuda


In [6]:
import os, json, time, random, csv
import numpy as np
from pathlib import Path
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
import torch
import torch.nn as nn

RUN_NAME = "dgrn_5m_v2_local_4gb_run1"
RUN_DIR = RUNS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_RUN_NAME = "dgrn_5m_v2_run1"
INIT_CKPT_PATH = RUNS_ROOT / SOURCE_RUN_NAME / "ckpt_best.pt"

CKPT_PATH = RUN_DIR / "ckpt_latest.pt"
CKPT_BEST_PATH = RUN_DIR / "ckpt_best.pt"
HIST_JSON = RUN_DIR / "history.json"
HIST_CSV = RUN_DIR / "history.csv"

print("RUN_DIR:", RUN_DIR)
print("INIT_CKPT_PATH:", INIT_CKPT_PATH)

EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-5
GRAD_CLIP_NORM = 1.0
GRAD_ACCUM_STEPS = int(GRAD_ACCUM_STEPS_DEFAULT if "GRAD_ACCUM_STEPS_DEFAULT" in globals() else 1)
EFFECTIVE_BATCH_SIZE = int(BATCH_SIZE * GRAD_ACCUM_STEPS)
LOG_EVERY = 200
VAL_MAX_SAMPLES = 500_000
TARGET_CLAMP_EPS = 1e-3
BEST_EPS = 1e-5

SAVE_LATEST_EVERY_STEPS = 400
SAVE_HISTORY_EVERY_STEPS = 400
MAX_TRAIN_STEPS_PER_EPOCH = None

AUTO_RESUME = True
RESUME_STRICT_CONFIG = True
RESUME_FALLBACK_TO_BEST = False
ALLOW_OVERWRITE_EXISTING_RUN = False
INIT_FROM_EXTERNAL_CHECKPOINT = True
INIT_LOAD_BEST_VAL = True

TRAIN_TARGET_MODE = "logit_space"
REQUIRE_FORWARD_LOGITS = True

MODEL_CFG = {
    "num_blocks": 20,
    "hidden_dim": 256,
    "input_channels": 18,
    "drop_path_rate": 0.05,
    "output_mode": "tanh",
}

MODEL_CLASS_NAME = getattr(DGRNChessNet, "__name__", str(DGRNChessNet))
RUN_CONFIG = {
    "model_class_name": MODEL_CLASS_NAME,
    "model_cfg": MODEL_CFG,
    "train_target_mode": TRAIN_TARGET_MODE,
    "require_forward_logits": bool(REQUIRE_FORWARD_LOGITS),
    "target_clamp_eps": float(TARGET_CLAMP_EPS),
    "lr": float(LR),
    "min_lr": float(MIN_LR),
    "weight_decay": float(WEIGHT_DECAY),
    "optimizer_name": "AdamW",
    "scheduler_name": "CosineAnnealingLR",
    "grad_clip_norm": (None if GRAD_CLIP_NORM is None else float(GRAD_CLIP_NORM)),
    "micro_batch_size": int(BATCH_SIZE),
    "effective_batch_size": int(EFFECTIVE_BATCH_SIZE),
    "grad_accum_steps": int(GRAD_ACCUM_STEPS),
    "num_workers": int(NUM_WORKERS),
    "seed": int(SEED),
    "data_root_source": str(DATA_ROOT),
    "data_root_active": str(DATA_ROOT_ACTIVE),
    "train_samples": int(SHARDS["train"]["num_samples"]),
    "val_samples": int(SHARDS["val"]["num_samples"]),
    "test_samples": int(SHARDS["test"]["num_samples"]),
    "local_shuffle_block": int(LOCAL_SHUFFLE_BLOCK),
    "train_drop_last": bool(TRAIN_DROP_LAST),
    "save_latest_every_steps": int(SAVE_LATEST_EVERY_STEPS),
    "save_history_every_steps": int(SAVE_HISTORY_EVERY_STEPS),
    "allow_overwrite_existing_run": bool(ALLOW_OVERWRITE_EXISTING_RUN),
    "resume_logic_version": 6,
}

CANONICAL_RESUME_KEYS = (
    "model_class_name",
    "model_cfg",
    "train_target_mode",
    "require_forward_logits",
    "target_clamp_eps",
    "lr",
    "min_lr",
    "weight_decay",
    "optimizer_name",
    "scheduler_name",
    "grad_clip_norm",
    "micro_batch_size",
    "effective_batch_size",
    "grad_accum_steps",
    "seed",
    "train_samples",
    "val_samples",
    "test_samples",
    "local_shuffle_block",
    "train_drop_last",
)

print("micro_batch_size:", BATCH_SIZE)
print("effective_batch_size:", EFFECTIVE_BATCH_SIZE)
print("grad_accum_steps:", GRAD_ACCUM_STEPS)

if TRAIN_TARGET_MODE not in {"logit_space", "y_space"}:
    raise ValueError(f"Unsupported TRAIN_TARGET_MODE: {TRAIN_TARGET_MODE}")

if (not AUTO_RESUME) and (not ALLOW_OVERWRITE_EXISTING_RUN):
    existing_artifacts = [p for p in (CKPT_PATH, CKPT_BEST_PATH, HIST_JSON, HIST_CSV) if p.exists()]
    if existing_artifacts:
        raise RuntimeError(
            "RUN_DIR already contains artifacts while AUTO_RESUME=False. "
            "Use a new RUN_NAME or set ALLOW_OVERWRITE_EXISTING_RUN=True intentionally. "
            f"Found: {[str(p.name) for p in existing_artifacts]}"
        )


def build_optimizer(model, lr: float, weight_decay: float):
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        name_l = name.lower()
        if param.ndim <= 1 or name.endswith(".bias") or ".bn" in name_l or "norm" in name_l:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    print(
        f"optimizer groups: decay={len(decay_params)} params, "
        f"no_decay={len(no_decay_params)} params, weight_decay={weight_decay}"
    )
    return AdamW(
        [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=lr,
    )


model = DGRNChessNet(**MODEL_CFG).to(device)

if TRAIN_TARGET_MODE == "logit_space" and REQUIRE_FORWARD_LOGITS and not hasattr(model, "forward_logits"):
    raise RuntimeError(
        "Current model does not implement forward_logits(). "
        "Update the import cell to your architecture_v2 model before training."
    )

train_loss_fn = nn.MSELoss()
metric_loss_fn = nn.MSELoss()
optimizer = build_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)

use_amp = (device.type == "cuda")
scaler = GradScaler(enabled=use_amp)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, EPOCHS),
    eta_min=MIN_LR,
)

DETERMINISTIC = False
torch.backends.cudnn.benchmark = (not DETERMINISTIC)
torch.backends.cudnn.deterministic = DETERMINISTIC
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

history = {
    "epoch": [],
    "train_loss": [],
    "train_objective": [],
    "val_loss": [],
    "lr": [],
    "train_steps_logged": [],
}


def _canonical_resume_config(cfg: dict):
    if not isinstance(cfg, dict):
        return None
    return {key: cfg.get(key) for key in CANONICAL_RESUME_KEYS}


def _config_equal(a: dict, b: dict) -> bool:
    return json.dumps(_canonical_resume_config(a), sort_keys=True) == json.dumps(_canonical_resume_config(b), sort_keys=True)


def _config_diff(a: dict, b: dict):
    diffs = []
    left = _canonical_resume_config(a) or {}
    right = _canonical_resume_config(b) or {}
    for key in CANONICAL_RESUME_KEYS:
        if left.get(key) != right.get(key):
            diffs.append((key, left.get(key), right.get(key)))
    return diffs


def _atomic_json_dump(obj, path: Path):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)
    os.replace(tmp, path)


def _atomic_write_lines(lines, path: Path):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.writelines(lines)
    os.replace(tmp, path)


def _atomic_torch_save(payload, path: Path):
    tmp = Path(str(path) + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def _capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state


def _restore_rng_state(state):
    if not state:
        return
    try:
        if "python" in state:
            random.setstate(state["python"])
        if "numpy" in state:
            np.random.set_state(state["numpy"])
        if "torch_cpu" in state:
            torch.set_rng_state(state["torch_cpu"])
        if torch.cuda.is_available() and ("torch_cuda" in state):
            torch.cuda.set_rng_state_all(state["torch_cuda"])
    except Exception as e:
        print(f"[warn] Cannot restore RNG state: {e}")


def target_to_logits(y, eps: float = TARGET_CLAMP_EPS):
    y = torch.clamp(y, -1.0 + eps, 1.0 - eps)
    return torch.atanh(y)


def forward_logits(x):
    if hasattr(model, "forward_logits"):
        return model.forward_logits(x).view(-1)
    if REQUIRE_FORWARD_LOGITS:
        raise RuntimeError(
            "TRAIN_TARGET_MODE='logit_space' requires model.forward_logits(). "
            "Update the import cell to architecture_v2 before training."
        )
    pred = model(x).view(-1)
    pred = torch.clamp(pred, -1.0 + TARGET_CLAMP_EPS, 1.0 - TARGET_CLAMP_EPS)
    return torch.atanh(pred)


def compute_objective_and_metric(x, y):
    if TRAIN_TARGET_MODE == "logit_space":
        y_logits = target_to_logits(y)
        logits = forward_logits(x)
        pred_metric = torch.tanh(logits)
        objective = train_loss_fn(logits, y_logits)
        metric = metric_loss_fn(pred_metric, y)
        return objective, metric, pred_metric
    pred = model(x).view(-1)
    objective = train_loss_fn(pred, y)
    metric = metric_loss_fn(pred, y)
    return objective, metric, pred


def save_history():
    _atomic_json_dump(history, HIST_JSON)
    lines = ["epoch,train_loss,train_objective,val_loss,lr\n"]
    for e, tl, tobj, vl, lr_now in zip(
        history["epoch"],
        history["train_loss"],
        history["train_objective"],
        history["val_loss"],
        history["lr"],
    ):
        lines.append(f"{e},{tl:.8f},{tobj:.8f},{vl:.8f},{lr_now:.8g}\n")
    _atomic_write_lines(lines, HIST_CSV)


@torch.no_grad()
def evaluate(loader, max_samples=None):
    model.eval()
    total_metric = 0.0
    total_n = 0
    for x, y in loader:
        if (max_samples is not None) and (total_n >= max_samples):
            break
        x = x.to(device, non_blocking=True, dtype=torch.float32)
        y = y.to(device, non_blocking=True).float().view(-1)
        if max_samples is not None:
            remain = int(max_samples - total_n)
            if remain <= 0:
                break
            if y.numel() > remain:
                x = x[:remain]
                y = y[:remain]
        with autocast(device_type=device.type, enabled=use_amp):
            _, metric, _ = compute_objective_and_metric(x, y)
        bs = y.numel()
        total_metric += float(metric.item()) * bs
        total_n += bs
    return total_metric / max(1, total_n)


def save_checkpoint(path: Path, epoch: int, global_step: int, best_val: float, last_val=None, tag="latest", epoch_step: int = 0, is_epoch_end: bool = True, epoch_state=None):
    payload = {
        "epoch": int(epoch),
        "epoch_step": int(epoch_step),
        "is_epoch_end": bool(is_epoch_end),
        "global_step": int(global_step),
        "best_val": float(best_val) if best_val != float("inf") else None,
        "last_val": (None if last_val is None else float(last_val)),
        "tag": str(tag),
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict() if use_amp else None,
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "history": history,
        "config": RUN_CONFIG,
        "rng_state": _capture_rng_state(),
        "epoch_state": epoch_state or {},
    }
    _atomic_torch_save(payload, path)


def _load_checkpoint_from(path: Path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    ckpt["__path"] = str(path)
    return ckpt


def _init_from_external_checkpoint(path: Path):
    ckpt = _load_checkpoint_from(path)
    model.load_state_dict(ckpt["model"], strict=True)
    init_best = ckpt.get("best_val", None)
    if (init_best is None) or (not INIT_LOAD_BEST_VAL):
        init_best = float("inf")
    print(f"Initialized model weights from external checkpoint: {path}")
    return 0, 0, 0, float(init_best), None


def try_resume():
    if AUTO_RESUME and CKPT_PATH.exists():
        ckpt = _load_checkpoint_from(CKPT_PATH)
        ckpt_cfg = ckpt.get("config")
        if RESUME_STRICT_CONFIG:
            if ckpt_cfg is None:
                raise RuntimeError("Checkpoint missing config metadata while RESUME_STRICT_CONFIG=True")
            if not _config_equal(ckpt_cfg, RUN_CONFIG):
                diffs = _config_diff(ckpt_cfg, RUN_CONFIG)
                diff_lines = [f"{key}: checkpoint={left!r} current={right!r}" for key, left, right in diffs]
                raise RuntimeError(
                    "Checkpoint config mismatch on canonical training keys. "
                    "Use a new RUN_NAME or intentionally change the current config.\n"
                    + "\n".join(diff_lines)
                )
        model.load_state_dict(ckpt["model"], strict=True)
        if ckpt.get("optimizer") is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if use_amp and ckpt.get("scaler") is not None:
            scaler.load_state_dict(ckpt["scaler"])
        if scheduler is not None and ckpt.get("scheduler") is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        ckpt_history = ckpt.get("history")
        if isinstance(ckpt_history, dict):
            for key in history.keys():
                if isinstance(ckpt_history.get(key), list):
                    history[key] = ckpt_history[key]
        resumed_best = ckpt.get("best_val", None)
        if resumed_best is None:
            resumed_best = min(history["val_loss"]) if len(history["val_loss"]) > 0 else float("inf")
        saved_epoch = int(ckpt.get("epoch", 0))
        saved_epoch_step = int(ckpt.get("epoch_step", 0))
        is_epoch_end = bool(ckpt.get("is_epoch_end", True))
        if is_epoch_end:
            start_epoch = saved_epoch + 1
            start_step_in_epoch = 0
            resume_epoch_state = None
        else:
            start_epoch = saved_epoch
            start_step_in_epoch = max(0, saved_epoch_step)
            resume_epoch_state = ckpt.get("epoch_state", None)
        resumed_global_step = int(ckpt.get("global_step", 0))
        _restore_rng_state(ckpt.get("rng_state", None))
        print(
            f"Resumed local run from {ckpt.get('__path', CKPT_PATH)}: epoch={saved_epoch}, "
            f"step={saved_epoch_step}, is_epoch_end={is_epoch_end}, global_step={resumed_global_step}"
        )
        return start_epoch, start_step_in_epoch, resumed_global_step, float(resumed_best), resume_epoch_state

    if INIT_FROM_EXTERNAL_CHECKPOINT and INIT_CKPT_PATH.exists():
        return _init_from_external_checkpoint(INIT_CKPT_PATH)

    print("[resume] No checkpoint found, starting from scratch.")
    return 0, 0, 0, float("inf"), None


start_epoch, start_step_in_epoch, global_step, best_val, resumed_epoch_state = try_resume()
if best_val == float("inf") and len(history["val_loss"]) > 0:
    best_val = min(history["val_loss"])


for epoch in range(start_epoch, EPOCHS):
    resume_batch_offset = start_step_in_epoch if epoch == start_epoch else 0
    if "train_batch_sampler" in globals() and hasattr(train_batch_sampler, "set_epoch"):
        train_batch_sampler.set_epoch(epoch)
    if "train_batch_sampler" in globals() and hasattr(train_batch_sampler, "set_start_batch"):
        train_batch_sampler.set_start_batch(resume_batch_offset)
    if resume_batch_offset > 0:
        print(f"[resume] epoch={epoch}: continue from micro-batch {resume_batch_offset}")
    if (epoch == start_epoch) and (resumed_epoch_state is not None) and (resume_batch_offset > 0):
        running_metric = float(resumed_epoch_state.get("running_metric", 0.0))
        running_objective = float(resumed_epoch_state.get("running_objective", 0.0))
        running_n = int(resumed_epoch_state.get("running_n", 0))
    else:
        running_metric = 0.0
        running_objective = 0.0
        running_n = 0

    model.train()
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)
    train_loader_len = len(train_loader)

    for step, (x, y) in enumerate(train_loader):
        step_abs = step + resume_batch_offset
        if (MAX_TRAIN_STEPS_PER_EPOCH is not None) and (step_abs >= MAX_TRAIN_STEPS_PER_EPOCH):
            break
        x = x.to(device, non_blocking=True, dtype=torch.float32)
        y = y.to(device, non_blocking=True).float().view(-1)
        with autocast(device_type=device.type, enabled=use_amp):
            objective, metric, _ = compute_objective_and_metric(x, y)
            objective_scaled = objective / GRAD_ACCUM_STEPS
        if not torch.isfinite(objective):
            print(f"[warn] non-finite objective at epoch={epoch} micro_step={step_abs}, skip batch")
            optimizer.zero_grad(set_to_none=True)
            continue
        if use_amp:
            scaler.scale(objective_scaled).backward()
        else:
            objective_scaled.backward()

        is_last_micro = (step == train_loader_len - 1)
        should_step = (((step_abs + 1) % GRAD_ACCUM_STEPS) == 0) or is_last_micro

        bs = y.numel()
        batch_metric = float(metric.item())
        batch_objective = float(objective.item())
        running_metric += batch_metric * bs
        running_objective += batch_objective * bs
        running_n += bs

        if should_step:
            if use_amp:
                if GRAD_CLIP_NORM is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % LOG_EVERY == 0:
                avg_metric = running_metric / max(1, running_n)
                avg_objective = running_objective / max(1, running_n)
                history["train_steps_logged"].append([
                    global_step,
                    float(avg_metric),
                    batch_metric,
                    float(avg_objective),
                    batch_objective,
                ])
                lr_now = optimizer.param_groups[0]["lr"]
                print(
                    f"epoch={epoch} micro_step={step_abs} global_step={global_step} lr={lr_now:.3g} "
                    f"train_metric(batch)={batch_metric:.6f} train_metric(avg)={avg_metric:.6f} "
                    f"train_obj(batch)={batch_objective:.6f} train_obj(avg)={avg_objective:.6f}"
                )

            if (SAVE_HISTORY_EVERY_STEPS is not None) and (global_step % SAVE_HISTORY_EVERY_STEPS == 0):
                save_history()

            if (SAVE_LATEST_EVERY_STEPS is not None) and (global_step % SAVE_LATEST_EVERY_STEPS == 0):
                save_checkpoint(
                    CKPT_PATH,
                    epoch,
                    global_step,
                    best_val=best_val,
                    last_val=None,
                    tag="step_latest",
                    epoch_step=step_abs + 1,
                    is_epoch_end=False,
                    epoch_state={
                        "running_metric": float(running_metric),
                        "running_objective": float(running_objective),
                        "running_n": int(running_n),
                    },
                )

    train_loss_epoch = running_metric / max(1, running_n)
    train_objective_epoch = running_objective / max(1, running_n)
    val_loss_epoch = evaluate(val_loader, max_samples=VAL_MAX_SAMPLES)

    if scheduler is not None:
        scheduler.step()

    lr_now = optimizer.param_groups[0]["lr"]
    dt = time.time() - t0

    history["epoch"].append(int(epoch))
    history["train_loss"].append(float(train_loss_epoch))
    history["train_objective"].append(float(train_objective_epoch))
    history["val_loss"].append(float(val_loss_epoch))
    history["lr"].append(float(lr_now))

    improved = (best_val == float("inf")) or (val_loss_epoch < best_val - BEST_EPS)
    if improved:
        best_val = float(val_loss_epoch)
        save_checkpoint(
            CKPT_BEST_PATH,
            epoch,
            global_step,
            best_val=best_val,
            last_val=val_loss_epoch,
            tag="best",
            epoch_step=0,
            is_epoch_end=True,
            epoch_state={},
        )

    print(
        f"[epoch {epoch}] train_metric={train_loss_epoch:.6f} train_obj={train_objective_epoch:.6f} "
        f"val_loss={val_loss_epoch:.6f} lr={lr_now:.3g} time={dt:.1f}s {'(best)' if improved else ''}"
    )

    save_history()
    save_checkpoint(
        CKPT_PATH,
        epoch,
        global_step,
        best_val=best_val,
        last_val=val_loss_epoch,
        tag="epoch_latest",
        epoch_step=0,
        is_epoch_end=True,
        epoch_state={},
    )

print("Done. Logs saved to:", HIST_CSV, "and", HIST_JSON)
print("Latest checkpoint:", CKPT_PATH)
print("Best checkpoint:", CKPT_BEST_PATH)


RUN_DIR: C:\Users\USER\Desktop\chess_engine\runs\dgrn_5m_v2_local_4gb_run1
INIT_CKPT_PATH: C:\Users\USER\Desktop\chess_engine\runs\dgrn_5m_v2_run1\ckpt_best.pt
micro_batch_size: 128
effective_batch_size: 2048
grad_accum_steps: 16
optimizer groups: decay=131 params, no_decay=235 params, weight_decay=0.0001
Initialized model weights from external checkpoint: C:\Users\USER\Desktop\chess_engine\runs\dgrn_5m_v2_run1\ckpt_best.pt
epoch=0 micro_step=3199 global_step=200 lr=0.0001 train_metric(batch)=0.077445 train_metric(avg)=0.079901 train_obj(batch)=0.107122 train_obj(avg)=0.217133
epoch=0 micro_step=6399 global_step=400 lr=0.0001 train_metric(batch)=0.066620 train_metric(avg)=0.080341 train_obj(batch)=0.177634 train_obj(avg)=0.218326
epoch=0 micro_step=9599 global_step=600 lr=0.0001 train_metric(batch)=0.087308 train_metric(avg)=0.080221 train_obj(batch)=0.230459 train_obj(avg)=0.219083
epoch=0 micro_step=12799 global_step=800 lr=0.0001 train_metric(batch)=0.056017 train_metric(avg)=0.0801

KeyboardInterrupt: 

In [ ]:
# Plot epoch-level and step-level history from the active RUN_DIR.
import csv
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def resolve_run_dir(default_run_name="dgrn_5m_v2_local_4gb_run1"):
    if "RUN_DIR" in globals():
        return Path(RUN_DIR)
    if "RUNS_ROOT" in globals():
        return Path(RUNS_ROOT) / default_run_name

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        run_dir = candidate / "runs" / default_run_name
        if run_dir.exists():
            return run_dir
    raise FileNotFoundError(f"Cannot resolve local run dir for {default_run_name}")


RUN_DIR = resolve_run_dir()
hist_csv = RUN_DIR / "history.csv"
hist_json = RUN_DIR / "history.json"
assert hist_csv.exists(), f"Missing history.csv: {hist_csv}"
assert hist_json.exists(), f"Missing history.json: {hist_json}"

epochs = []
train_loss = []
train_objective = []
val_loss = []
lr_list = []

with open(hist_csv, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs.append(int(row["epoch"]))
        train_loss.append(float(row["train_loss"]))
        train_objective.append(float(row.get("train_objective", row["train_loss"])))
        val_loss.append(float(row["val_loss"]))
        lr_list.append(float(row["lr"]))

plt.figure()
plt.plot(epochs, train_loss, label="train_metric_mse")
plt.plot(epochs, val_loss, label="val_metric_mse")
if len(train_objective) == len(epochs):
    plt.plot(epochs, train_objective, label="train_objective", linestyle="--")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Train/Val History")
plt.legend()
plt.tight_layout()
plt.show()

with open(hist_json, "r", encoding="utf-8") as f:
    hist = json.load(f)

steps = [int(s[0]) for s in hist.get("train_steps_logged", [])]
step_metric = [float(s[1]) for s in hist.get("train_steps_logged", [])]
step_objective = [float(s[3]) for s in hist.get("train_steps_logged", []) if len(s) > 3]

if len(steps) > 0:
    plt.figure()
    plt.plot(steps, step_metric, label="train_metric(avg window)")
    if len(step_objective) == len(steps):
        plt.plot(steps, step_objective, label="train_objective(avg window)", linestyle="--")

    win = min(50, max(1, len(step_metric) // 20))
    if win >= 3:
        x = np.array(steps, dtype=np.float64)
        y = np.array(step_metric, dtype=np.float64)
        y_ma = np.convolve(y, np.ones(win) / win, mode="valid")
        x_ma = x[win - 1 :]
        plt.plot(x_ma, y_ma, label=f"metric_moving_avg (win={win})")

    plt.xlabel("global_step")
    plt.ylabel("loss")
    plt.title("Step-level Train History")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No step-level logs found in history.json (train_steps_logged is empty).")


In [ ]:
# ===== Cell: Load checkpoint for evaluation =====
from pathlib import Path
import torch


def resolve_run_dir(default_run_name="dgrn_5m_v2_local_4gb_run1"):
    if "RUN_DIR" in globals():
        return Path(RUN_DIR)
    if "RUNS_ROOT" in globals():
        return Path(RUNS_ROOT) / default_run_name

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        run_dir = candidate / "runs" / default_run_name
        if run_dir.exists():
            return run_dir
    raise FileNotFoundError(f"Cannot resolve local run dir for {default_run_name}")


RUN_DIR = resolve_run_dir()
CKPT_LATEST_PATH = RUN_DIR / "ckpt_latest.pt"
CKPT_BEST_PATH = RUN_DIR / "ckpt_best.pt"
LOAD_BEST_FOR_EVAL = True

if LOAD_BEST_FOR_EVAL and CKPT_BEST_PATH.exists():
    CKPT_PATH = CKPT_BEST_PATH
else:
    CKPT_PATH = CKPT_LATEST_PATH

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert "DGRNChessNet" in globals(), "Model class is not imported. Run the import cell first."

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
ckpt_cfg = ckpt.get("config") or {}
ckpt_model_cfg = ckpt_cfg.get("model_cfg")
if ckpt_model_cfg is None:
    raise RuntimeError("Checkpoint is missing config['model_cfg']; cannot instantiate model safely.")

expected_model_class = ckpt_cfg.get("model_class_name")
current_model_class = getattr(DGRNChessNet, "__name__", str(DGRNChessNet))
if expected_model_class and expected_model_class != current_model_class:
    print(
        f"[warn] Imported model class is {current_model_class}, "
        f"but checkpoint expects {expected_model_class}."
    )

model = DGRNChessNet(**ckpt_model_cfg).to(device)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

print(f"Loaded checkpoint: {CKPT_PATH}")
print("device:", device)
print("checkpoint model_class_name:", expected_model_class)
print("checkpoint tag:", ckpt.get("tag"))
print("checkpoint epoch:", ckpt.get("epoch"))
print("checkpoint epoch_step:", ckpt.get("epoch_step"))
print("checkpoint is_epoch_end:", ckpt.get("is_epoch_end"))
print("checkpoint global_step:", ckpt.get("global_step"))
print("checkpoint best_val:", ckpt.get("best_val"))
print("config match:", expected_model_class == current_model_class)


In [ ]:
# ===== Cell: Bucketed Val Analysis (MSE per bucket + calibration) =====
import numpy as np
import torch
import matplotlib.pyplot as plt

# --- Config ---
# dùng đúng bucket edges như preprocessing (20 buckets)
N_BUCKETS = 20
BUCKET_EDGES = np.linspace(-1.0, 1.0, N_BUCKETS + 1)

# Giới hạn số mẫu để chạy nhanh; tăng dần lên 200k, 500k nếu muốn
MAX_VAL_SAMPLES = 200_000

@torch.no_grad()
def bucketed_val_analysis(model, loader, device, bucket_edges, max_samples=200_000):
    model.eval()

    n_buckets = len(bucket_edges) - 1
    counts = np.zeros(n_buckets, dtype=np.int64)

    # sum of squared error per bucket
    sse = np.zeros(n_buckets, dtype=np.float64)

    # sums for calibration
    sum_y = np.zeros(n_buckets, dtype=np.float64)
    sum_p = np.zeros(n_buckets, dtype=np.float64)

    # overall
    total_sse = 0.0
    total_n = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True, dtype=torch.float32) # Added dtype=torch.float32
        y = y.to(device, non_blocking=True).float().view(-1)

        # forward
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            p = model(x).view(-1)

        # move to cpu numpy
        y_np = y.detach().cpu().numpy()
        p_np = p.detach().cpu().numpy()

        # clip safety (tanh head thì gần như không cần, nhưng để chắc)
        y_np = np.clip(y_np, -1.0, 1.0)
        p_np = np.clip(p_np, -1.0, 1.0)

        err = p_np - y_np
        se = err * err

        # bucket id theo target y (khuyến nghị); bạn có thể bucket theo pred để phân tích khác
        bid = np.searchsorted(bucket_edges, y_np, side="right") - 1
        bid = np.clip(bid, 0, n_buckets - 1)

        # accumulate per bucket
        for b in range(n_buckets):
            mask = (bid == b)
            if not np.any(mask):
                continue
            c = int(mask.sum())
            counts[b] += c
            sse[b] += float(se[mask].sum())
            sum_y[b] += float(y_np[mask].sum())
            sum_p[b] += float(p_np[mask].sum())

        total_sse += float(se.sum())
        total_n += int(y_np.size)

        if total_n >= max_samples:
            break

    mse_total = total_sse / max(1, total_n)
    mse_bucket = np.divide(sse, np.maximum(1, counts))
    mean_y = np.divide(sum_y, np.maximum(1, counts))
    mean_p = np.divide(sum_p, np.maximum(1, counts))

    out = {
        "n_used": total_n,
        "mse_total": mse_total,
        "counts": counts,
        "mse_bucket": mse_bucket,
        "mean_y": mean_y,
        "mean_p": mean_p,
    }
    return out

# ---- Run analysis ----
res = bucketed_val_analysis(model, val_loader, device, BUCKET_EDGES, max_samples=MAX_VAL_SAMPLES)

print(f"Val bucketed analysis: n_used={res['n_used']}, mse_total={res['mse_total']:.6f}")
print("counts:", res["counts"].tolist())
print("mse_bucket:", [float(f"{v:.6f}") for v in res["mse_bucket"]])

# ---- Plot 1: val distribution by bucket ----
bucket_centers = 0.5 * (BUCKET_EDGES[:-1] + BUCKET_EDGES[1:])

plt.figure()
plt.bar(bucket_centers, res["counts"], width=(BUCKET_EDGES[1]-BUCKET_EDGES[0])*0.9)
plt.xlabel("y bucket center")
plt.ylabel("count")
plt.title(f"Val distribution by bucket (n_used={res['n_used']})")
plt.tight_layout()
plt.show()

# ---- Plot 2: MSE by bucket ----
plt.figure()
plt.plot(bucket_centers, res["mse_bucket"], marker="o")
plt.xlabel("y bucket center")
plt.ylabel("MSE")
plt.title("Val MSE by bucket (bucketed by target y)")
plt.tight_layout()
plt.show()

# ---- Plot 3: Calibration (mean pred vs mean target per bucket) ----
plt.figure()
plt.plot(bucket_centers, res["mean_y"], marker="o", label="mean target y")
plt.plot(bucket_centers, res["mean_p"], marker="o", label="mean pred")
plt.xlabel("y bucket center")
plt.ylabel("value")
plt.title("Calibration per bucket (mean pred vs mean target)")
plt.legend()
plt.tight_layout()
plt.show()

# ---- Optional: print worst buckets ----
worst = np.argsort(-res["mse_bucket"])[:5]
print("Worst 5 buckets (by MSE):")
for b in worst:
    print(f"  bucket {b:02d} [{BUCKET_EDGES[b]:.2f},{BUCKET_EDGES[b+1]:.2f}] "
          f"count={int(res['counts'][b])} mse={res['mse_bucket'][b]:.6f} "
          f"mean_y={res['mean_y'][b]:.3f} mean_p={res['mean_p'][b]:.3f}")

In [ ]:
# ===== Cell: Test evaluation metrics (MSE/MAE/R2 + correlations + "accuracy" for regression) =====
import numpy as np
import torch
import matplotlib.pyplot as plt

# --- Config ---
MAX_TEST_SAMPLES = 200_000   # tăng lên 500_000 nếu bạn muốn full test
STATIC_THRESH = 0.20         # vùng "tĩnh" |y| <= 0.2
DEADZONE = 0.02              # vùng coi như hòa (bỏ qua khi tính sign-acc)
EPS_LIST = [0.05, 0.10, 0.20]

N_BUCKETS = 20
BUCKET_EDGES = np.linspace(-1.0, 1.0, N_BUCKETS + 1)

@torch.no_grad()
def collect_preds(model, loader, device, max_samples=200_000):
    model.eval()
    ys = []
    ps = []
    n = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float().view(-1)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            p = model(x).view(-1)

        y_np = y.detach().cpu().numpy()
        p_np = p.detach().cpu().numpy()

        ys.append(y_np)
        ps.append(p_np)
        n += y_np.size
        if n >= max_samples:
            break

    y_all = np.concatenate(ys, axis=0)[:max_samples]
    p_all = np.concatenate(ps, axis=0)[:max_samples]
    # safety clip
    y_all = np.clip(y_all, -1.0, 1.0)
    p_all = np.clip(p_all, -1.0, 1.0)
    return y_all.astype(np.float64), p_all.astype(np.float64)

def r2_score(y, p):
    ss_res = np.sum((y - p) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1.0 - ss_res / (ss_tot + 1e-12)

def pearsonr(y, p):
    y0 = y - np.mean(y)
    p0 = p - np.mean(p)
    num = np.sum(y0 * p0)
    den = np.sqrt(np.sum(y0*y0) * np.sum(p0*p0)) + 1e-12
    return num / den

def spearmanr(y, p):
    # rank via argsort twice (ties không xử lý hoàn hảo, nhưng đủ cho debug nhanh)
    ry = np.empty_like(y, dtype=np.int64)
    rp = np.empty_like(p, dtype=np.int64)
    ry[np.argsort(y)] = np.arange(y.size)
    rp[np.argsort(p)] = np.arange(p.size)
    return pearsonr(ry.astype(np.float64), rp.astype(np.float64))

def bucket_id(v, edges):
    b = np.searchsorted(edges, v, side="right") - 1
    return np.clip(b, 0, len(edges)-2)

def eval_metrics(y, p, name="all"):
    err = p - y
    mse = float(np.mean(err**2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(err)))
    r2 = float(r2_score(y, p))
    pr = float(pearsonr(y, p))
    sr = float(spearmanr(y, p))

    # "Accuracy" variants (regression-friendly)
    # 1) Within-epsilon accuracy
    within_eps = {eps: float(np.mean(np.abs(err) <= eps)) for eps in EPS_LIST}

    # 2) Sign accuracy with deadzone around 0
    y_sign = np.sign(y)
    p_sign = np.sign(p)
    mask = (np.abs(y) > DEADZONE)  # ignore near-0
    sign_acc = float(np.mean((y_sign[mask] == p_sign[mask]).astype(np.float64))) if np.any(mask) else float("nan")
    kept_ratio = float(np.mean(mask.astype(np.float64)))

    # 3) Bucket accuracy (20 buckets)
    yb = bucket_id(y, BUCKET_EDGES)
    pb = bucket_id(p, BUCKET_EDGES)
    bucket_acc = float(np.mean((yb == pb).astype(np.float64)))

    # 4) "Directional correctness" on non-trivial positions (|y| >= STATIC_THRESH)
    hard = (np.abs(y) >= STATIC_THRESH)
    hard_sign_acc = float(np.mean((y_sign[hard] == p_sign[hard]).astype(np.float64))) if np.any(hard) else float("nan")

    out = {
        "name": name,
        "n": int(y.size),
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "pearson": pr,
        "spearman": sr,
        "bucket_acc": bucket_acc,
        "sign_acc": sign_acc,
        "sign_acc_kept_ratio": kept_ratio,
        "hard_sign_acc(|y|>=t)": hard_sign_acc,
        "within_eps": within_eps,
    }
    return out

# ---- Collect predictions on test ----
y_test, p_test = collect_preds(model, test_loader, device, max_samples=MAX_TEST_SAMPLES)

# ---- Metrics overall ----
m_all = eval_metrics(y_test, p_test, name="test_all")

# ---- Metrics on "static" subset (|y| <= STATIC_THRESH) ----
mask_static = (np.abs(y_test) <= STATIC_THRESH)
m_static = eval_metrics(y_test[mask_static], p_test[mask_static], name=f"test_static(|y|<={STATIC_THRESH})")

# ---- Metrics on "non-static" subset (|y| > STATIC_THRESH) ----
mask_dyn = ~mask_static
m_dyn = eval_metrics(y_test[mask_dyn], p_test[mask_dyn], name=f"test_dynamic(|y|>{STATIC_THRESH})")

def print_metrics(m):
    print(f"\n== {m['name']} ==")
    print("n:", m["n"])
    print(f"MSE={m['mse']:.6f} RMSE={m['rmse']:.4f} MAE={m['mae']:.4f}")
    print(f"R2={m['r2']:.4f} Pearson={m['pearson']:.4f} Spearman={m['spearman']:.4f}")
    print(f"BucketAcc(20)={m['bucket_acc']:.4f}")
    print(f"SignAcc(deadzone={DEADZONE})={m['sign_acc']:.4f}  kept={m['sign_acc_kept_ratio']:.3f}")
    print(f"HardSignAcc(|y|>={STATIC_THRESH})={m['hard_sign_acc(|y|>=t)']:.4f}")
    for eps, acc in m["within_eps"].items():
        print(f"Within±{eps:.2f}: {acc:.4f}")

print_metrics(m_all)
print_metrics(m_static)
print_metrics(m_dyn)

# ---- Optional quick plots: scatter + error vs |y| ----
# Scatter (subsample)
sub = min(20000, y_test.size)
idx = np.random.default_rng(0).choice(y_test.size, size=sub, replace=False)

plt.figure()
plt.scatter(y_test[idx], p_test[idx], s=2)
plt.xlabel("target y")
plt.ylabel("pred")
plt.title("Test scatter: pred vs target (subsample)")
plt.tight_layout()
plt.show()

plt.figure()
plt.scatter(np.abs(y_test[idx]), np.abs(p_test[idx]-y_test[idx]), s=2)
plt.xlabel("|target y|")
plt.ylabel("|error|")
plt.title("Test error magnitude vs |target| (subsample)")
plt.tight_layout()
plt.show()
